In [21]:
import asyncio
import json
import time
import uuid
from collections.abc import Awaitable, Callable
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, Field, ValidationError, model_validator

### Task 1

In [2]:
class SubTask(BaseModel):
    id: str
    objective: str
    output_format: str = "text"
    boundaries: str = ""
    max_steps: int = Field(default=6, ge=1, le=15)


class WorkerResult(BaseModel):
    task_id: str
    status: Literal["ok", "failed", "timeout"]
    summary: str = ""
    sources: list[str] = Field(default_factory=list)
    steps_used: int = Field(default=0, ge=0)
    tokens_used: int = Field(default=0, ge=0)

    @model_validator(mode="after")
    def validate_summary_on_ok(self) -> "WorkerResult":
        if self.status == "ok" and not self.summary.strip():
            raise ValueError("Summary cannot be empty when status is ok")
        return self

In [3]:
# 1. Valid instances
task = SubTask(id="sub-1", objective="Summarize research", max_steps=5)
result = WorkerResult(task_id="sub-1", status="ok", summary="Found key findings.")
print("Valid task:", task)
print("Valid result:", result)

# 2. Bound validation (max_steps must be 1 <= max_steps <= 15)
try:
    SubTask(id="sub-2", objective="Test bounds", max_steps=20)
except ValidationError as e:
    print("Expected bound error:", e.errors()[0]["msg"])

# 3. Status validation
try:
    WorkerResult(task_id="sub-2", status="unknown")  # type: ignore[arg-type]
except ValidationError as e:
    print("Expected status error:", e.errors()[0]["msg"])

# 4. Summary required on status=ok
try:
    WorkerResult(task_id="sub-2", status="ok", summary="")
except ValidationError as e:
    print("Expected summary error:", e.errors()[0]["msg"])

Valid task: id='sub-1' objective='Summarize research' output_format='text' boundaries='' max_steps=5
Valid result: task_id='sub-1' status='ok' summary='Found key findings.' sources=[] steps_used=0 tokens_used=0
Expected bound error: Input should be less than or equal to 15
Expected status error: Input should be 'ok', 'failed' or 'timeout'
Expected summary error: Value error, Summary cannot be empty when status is ok


### Task 2

In [4]:
async def run_worker(
    task: SubTask,
    worker_fn: Callable[[SubTask], Awaitable[WorkerResult]],
    timeout: float = 5.0,
) -> WorkerResult:
    try:
        async with asyncio.timeout(timeout):
            return await worker_fn(task)
    except TimeoutError:
        return WorkerResult(
            task_id=task.id,
            status="timeout",
            summary=f"Worker timed out after {timeout}s",
        )
    except Exception as exc:  # noqa: BLE001
        return WorkerResult(
            task_id=task.id,
            status="failed",
            summary=f"Worker failed: {type(exc).__name__}: {exc}",
        )

In [5]:
async def ok_worker(t: SubTask) -> WorkerResult:
    return WorkerResult(task_id=t.id, status="ok", summary="Success")


async def crash_worker(t: SubTask) -> WorkerResult:
    raise RuntimeError("Database connection lost")


async def slow_worker(t: SubTask) -> WorkerResult:
    await asyncio.sleep(1.0)
    return WorkerResult(task_id=t.id, status="ok", summary="Done")


res_ok = await run_worker(task, ok_worker)
res_fail = await run_worker(task, crash_worker)
res_timeout = await run_worker(task, slow_worker, timeout=0.05)

print("OK result:", res_ok.status, "|", res_ok.summary)
print("Fail result:", res_fail.status, "|", res_fail.summary)
print("Timeout result:", res_timeout.status, "|", res_timeout.summary)

OK result: ok | Success
Fail result: failed | Worker failed: RuntimeError: Database connection lost
Timeout result: timeout | Worker timed out after 0.05s


### Task 3

In [6]:
async def orchestrate(
    tasks: list[SubTask],
    worker_fn: Callable[[SubTask], Awaitable[WorkerResult]],
    max_concurrency: int = 3,
    timeout: float = 5.0,
) -> list[WorkerResult]:
    sem = asyncio.Semaphore(max_concurrency)

    async def _bounded_worker(task: SubTask) -> WorkerResult:
        async with sem:
            return await run_worker(task, worker_fn, timeout=timeout)

    return list(await asyncio.gather(*(_bounded_worker(t) for t in tasks)))

In [7]:
subtasks = [
    SubTask(id="task-1", objective="Extract data"),
    SubTask(id="task-2", objective="Analyze trends"),
    SubTask(id="task-3", objective="Generate charts"),
    SubTask(id="task-4", objective="Review output"),
]


async def mixed_worker(t: SubTask) -> WorkerResult:
    if t.id == "task-2":
        raise ValueError("Corrupted data")
    if t.id == "task-3":
        await asyncio.sleep(0.5)
    return WorkerResult(task_id=t.id, status="ok", summary=f"Done {t.id}")


results = await orchestrate(subtasks, mixed_worker, max_concurrency=2, timeout=0.1)
for r in results:
    print(f"{r.task_id}: status={r.status} | {r.summary}")

task-1: status=ok | Done task-1
task-2: status=failed | Worker failed: ValueError: Corrupted data
task-3: status=timeout | Worker timed out after 0.1s
task-4: status=ok | Done task-4


### Task 4

In [8]:
class Budget:
    def __init__(self, limit: int):
        self.limit = limit
        self.used = 0
        self._lock = asyncio.Lock()

    async def charge(self, amount: int) -> bool:
        async with self._lock:
            if self.used + amount <= self.limit:
                self.used += amount
                return True
            return False

    @property
    def remaining(self) -> int:
        return max(0, self.limit - self.used)

In [9]:
# Concurrency test: 20 concurrent tasks each trying to charge 10 tokens against limit 100
budget = Budget(limit=100)


async def attempt_charge(idx: int) -> bool:
    await asyncio.sleep(0.01)
    return await budget.charge(10)


charge_results = await asyncio.gather(*(attempt_charge(i) for i in range(20)))

successful_charges = sum(charge_results)
failed_charges = len(charge_results) - successful_charges

print(f"Successful charges: {successful_charges}, Failed: {failed_charges}")
print(f"Budget used: {budget.used}/{budget.limit}, remaining: {budget.remaining}")

assert budget.used == 100, f"Expected 100 used, got {budget.used}"
assert successful_charges == 10, f"Expected 10 successes, got {successful_charges}"
assert failed_charges == 10, f"Expected 10 failures, got {failed_charges}"
print("Cap held strictly under concurrency!")

Successful charges: 10, Failed: 10
Budget used: 100/100, remaining: 0
Cap held strictly under concurrency!


### Task 5

In [10]:
def merge_results(
    results: list[WorkerResult],
    synthesizer: Callable[[list[str]], str] = "\n".join,
) -> tuple[str, list[str]]:
    valid = [r.summary for r in results if r.status == "ok"]
    gaps = [f"{r.task_id} ({r.status}): {r.summary}" for r in results if r.status != "ok"]
    return synthesizer(valid), gaps

In [11]:
sample_results = [
    WorkerResult(task_id="t1", status="ok", summary="Revenue grew 12% YoY."),
    WorkerResult(task_id="t2", status="failed", summary="Database connection lost."),
    WorkerResult(task_id="t3", status="ok", summary="Operating expenses down 4%."),
    WorkerResult(task_id="t4", status="timeout", summary="Worker timed out after 5s."),
]

merged, gaps = merge_results(sample_results)
print("Merged:\n" + merged)
print("\nGaps:", gaps)

assert "Revenue grew" in merged and "Operating expenses" in merged
assert "Database connection lost" not in merged
assert len(gaps) == 2

Merged:
Revenue grew 12% YoY.
Operating expenses down 4%.

Gaps: ['t2 (failed): Database connection lost.', 't4 (timeout): Worker timed out after 5s.']


### Task 6

In [15]:
def filter_handoff_history(
    messages: list[dict[str, Any]],
    summary: str = "",
    summarizer: Callable[[list[dict[str, Any]]], str] | None = None,
) -> list[dict[str, str]]:
    user_msgs: list[dict[str, str]] = [
        {"role": "user", "content": m["content"]} for m in messages if m.get("role") == "user"
    ]
    summary_text = summarizer(messages) if summarizer else summary
    if summary_text:
        user_msgs.append({"role": "system", "content": f"Context Summary: {summary_text}"})
    return user_msgs

In [16]:
raw_history = [
    {"role": "user", "content": "I need help with my monthly invoice."},
    {
        "role": "assistant",
        "content": "",
        "tool_calls": [{"name": "fetch_db", "args": {"id": 101}}],
    },
    {"role": "tool", "content": "{'status': 'error', 'secret_key': 'xyz123'}"},
    {"role": "assistant", "content": "Checking secondary backup records..."},
    {"role": "user", "content": "Specifically invoice #INV-9982."},
]

filtered = filter_handoff_history(
    raw_history,
    summary="User is inquiring about invoice #INV-9982 billing details.",
)

for m in filtered:
    print(f"[{m['role'].upper()}]: {m['content']}")

assert len(filtered) == 3
assert all(m["role"] in ("user", "system") for m in filtered)
assert not any("tool_calls" in m or "secret_key" in m["content"] for m in filtered)

[USER]: I need help with my monthly invoice.
[USER]: Specifically invoice #INV-9982.
[SYSTEM]: Context Summary: User is inquiring about invoice #INV-9982 billing details.


### Task 7

In [17]:
class DelegationGuard:
    def __init__(self, max_depth: int = 4):
        self.max_depth = max_depth
        self.history: list[tuple[str, str, str]] = []

    def check_and_add(self, from_agent: str, to_agent: str, task: str) -> None:
        triple = (from_agent, to_agent, task)
        if triple in self.history:
            raise RuntimeError(f"Delegation loop detected on {triple}")
        if len(self.history) >= self.max_depth:
            raise RuntimeError(f"Delegation depth limit ({self.max_depth}) exceeded")
        self.history.append(triple)

In [18]:
# 1. Test loop detection
guard_loop = DelegationGuard(max_depth=5)
guard_loop.check_and_add("triage", "billing", "Refund request")
guard_loop.check_and_add("billing", "manager", "Refund request")
try:
    guard_loop.check_and_add("triage", "billing", "Refund request")  # Repeated triple
except RuntimeError as exc:
    print("Loop caught:", exc)

# 2. Test depth limit
guard_depth = DelegationGuard(max_depth=3)
for i in range(3):
    guard_depth.check_and_add(f"agent_{i}", f"agent_{i + 1}", f"Subtask {i}")
try:
    guard_depth.check_and_add("agent_3", "agent_4", "Subtask 3")  # Exceeds max_depth=3
except RuntimeError as exc:
    print("Depth caught:", exc)

Loop caught: Delegation loop detected on ('triage', 'billing', 'Refund request')
Depth caught: Delegation depth limit (3) exceeded


### Task 8

In [22]:
class TraceLogger:
    def __init__(self, log_path: Path | str = "traces.jsonl"):
        self.log_path = Path(log_path)

    def record(
        self,
        run_id: str,
        agent_name: str,
        tokens: int,
        latency: float,
        parent_id: str | None = None,
        span_id: str | None = None,
    ) -> dict[str, Any]:
        entry = {
            "span_id": span_id or str(uuid.uuid4())[:8],
            "run_id": run_id,
            "parent_id": parent_id,
            "agent_name": agent_name,
            "tokens": tokens,
            "latency": round(latency, 4),
        }
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        return entry


async def trace_delegation(
    run_id: str,
    agent_name: str,
    fn: Callable[[], Awaitable[tuple[Any, int]]],
    tracer: TraceLogger,
    parent_id: str | None = None,
) -> tuple[Any, dict[str, Any]]:
    start = time.perf_counter()
    result, tokens = await fn()
    latency = time.perf_counter() - start
    record = tracer.record(
        run_id=run_id,
        agent_name=agent_name,
        tokens=tokens,
        latency=latency,
        parent_id=parent_id,
    )
    return result, record

In [23]:
log_file = Path("exercise_traces.jsonl")
if log_file.exists():
    log_file.unlink()

tracer = TraceLogger(log_file)
run_id = f"run_{str(uuid.uuid4())[:6]}"

# 1. Lead agent execution
lead_span = tracer.record(
    run_id=run_id,
    agent_name="lead_orchestrator",
    tokens=120,
    latency=0.45,
    parent_id=None,
)


# 2. Child delegated execution with trace_delegation wrapper
async def mock_worker_call() -> tuple[str, int]:
    await asyncio.sleep(0.05)
    return "Invoice #123 processed", 45


worker_res, child_span = await trace_delegation(
    run_id=run_id,
    agent_name="billing_agent",
    fn=mock_worker_call,
    tracer=tracer,
    parent_id=lead_span["span_id"],
)

# 3. Verify JSON Lines log file
records = [json.loads(line) for line in log_file.read_text().splitlines() if line]
for r in records:
    print(
        f"Span {r['span_id']}: parent={r['parent_id']}, agent={r['agent_name']}, tokens={r['tokens']}, latency={r['latency']}s"
    )

assert len(records) == 2
assert records[0]["parent_id"] is None
assert records[1]["parent_id"] == records[0]["span_id"]
print("Tracing helper validated successfully with reconstructable tree!")

Span c1e95dda: parent=None, agent=lead_orchestrator, tokens=120, latency=0.45s
Span afdbf634: parent=c1e95dda, agent=billing_agent, tokens=45, latency=0.0502s
Tracing helper validated successfully with reconstructable tree!


### Task 9

In [24]:
class RouteDecision(BaseModel):
    specialists: list[Literal["billing", "technical", "account"]] = Field(default_factory=list)
    confidence: float = 1.0


def route_request(
    query: str,
    classifier_fn: Callable[[str], RouteDecision | None] | None = None,
    fallback_agent: str = "general_support",
) -> list[str]:
    if classifier_fn is None:

        def default_classifier(text: str) -> RouteDecision:
            matches: list[Literal["billing", "technical", "account"]] = []
            lower = text.lower()
            if any(w in lower for w in ("bill", "pay", "charge", "refund")):
                matches.append("billing")
            if any(w in lower for w in ("error", "crash", "bug", "down")):
                matches.append("technical")
            if any(w in lower for w in ("login", "password", "account")):
                matches.append("account")
            return RouteDecision(specialists=matches)

        classifier_fn = default_classifier

    try:
        decision = classifier_fn(query)
        if decision and decision.specialists:
            return list(decision.specialists)
    except Exception:  # noqa: BLE001
        return [fallback_agent]
    return [fallback_agent]

In [25]:
# 1. Specialist routing
print("Billing query:", route_request("Why was I charged twice for subscription?"))
print("Multi-specialist query:", route_request("App crashed during payment processing."))

# 2. Fallback routing for unknown/unmatched request
print("Ambiguous query:", route_request("What is the weather in Tokyo?"))


# 3. Fallback on classifier error via dependency injection
def faulty_classifier(text: str) -> RouteDecision:
    raise ConnectionError("Classifier service unavailable")


print("Faulty classifier query:", route_request("Pay bill", classifier_fn=faulty_classifier))

assert route_request("Refund my charge") == ["billing"]
assert route_request("Crash bug") == ["technical"]
assert route_request("Hello there") == ["general_support"]
assert route_request("Pay bill", classifier_fn=faulty_classifier) == ["general_support"]

Billing query: ['billing']
Multi-specialist query: ['billing', 'technical']
Ambiguous query: ['general_support']
Faulty classifier query: ['general_support']


### Task 10

In [26]:
class EvalMetrics(BaseModel):
    pattern: str
    quality: float
    latency: float
    cost: float


async def evaluate_agents(
    questions: list[str],
    single_agent_fn: Callable[[str], Awaitable[tuple[str, int]]],
    multi_agent_fn: Callable[[str], Awaitable[tuple[str, int]]],
    scorer_fn: Callable[[str, str], float],
    price_per_token: float = 1.5e-7,
) -> list[tuple[str, EvalMetrics, EvalMetrics]]:
    results = []
    header_s = "Single Agent (Q / Lat / Cost)"
    header_m = "Multi Agent (Q / Lat / Cost)"
    print(f"{'Question':<30} | {header_s:<32} | {header_m:<32}")
    print("-" * 100)

    for q in questions:
        # Single agent
        t0 = time.perf_counter()
        ans_s, tok_s = await single_agent_fn(q)
        lat_s = time.perf_counter() - t0
        m_s = EvalMetrics(
            pattern="single",
            quality=scorer_fn(q, ans_s),
            latency=lat_s,
            cost=tok_s * price_per_token,
        )

        # Multi agent
        t0 = time.perf_counter()
        ans_m, tok_m = await multi_agent_fn(q)
        lat_m = time.perf_counter() - t0
        m_m = EvalMetrics(
            pattern="multi",
            quality=scorer_fn(q, ans_m),
            latency=lat_m,
            cost=tok_m * price_per_token,
        )

        results.append((q, m_s, m_m))
        print(
            f"{q[:28]:<30} | "
            f"Q={m_s.quality:.2f}  {m_s.latency:.2f}s  ${m_s.cost:.6f}       | "
            f"Q={m_m.quality:.2f}  {m_m.latency:.2f}s  ${m_m.cost:.6f}"
        )
    return results

In [27]:
eval_questions = [
    "How do I dispute an erroneous charge on my bill?",
    "App crashes with null pointer during database backup.",
    "How to migrate user authentication to OAuth 2.0?",
]


# Injected Mock Single Agent
async def mock_single(q: str) -> tuple[str, int]:
    await asyncio.sleep(0.02)
    return f"General answer for: {q}", 80


# Injected Mock Multi Agent (Specialist + Reviewer)
async def mock_multi(q: str) -> tuple[str, int]:
    await asyncio.sleep(0.05)
    return f"Detailed specialist resolution and verified solution for: {q}", 240


# Injected Quality Scorer
def mock_scorer(q: str, answer: str) -> float:
    return 0.95 if "verified solution" in answer else 0.70


eval_results = await evaluate_agents(
    questions=eval_questions,
    single_agent_fn=mock_single,
    multi_agent_fn=mock_multi,
    scorer_fn=mock_scorer,
)

assert len(eval_results) == 3
assert all(m_multi.quality > m_single.quality for _, m_single, m_multi in eval_results)
print("Evaluation comparison completed successfully!")

Question                       | Single Agent (Q / Lat / Cost)    | Multi Agent (Q / Lat / Cost)    
----------------------------------------------------------------------------------------------------
How do I dispute an erroneou   | Q=0.70  0.02s  $0.000012       | Q=0.95  0.05s  $0.000036
App crashes with null pointe   | Q=0.70  0.02s  $0.000012       | Q=0.95  0.05s  $0.000036
How to migrate user authenti   | Q=0.70  0.02s  $0.000012       | Q=0.95  0.05s  $0.000036
Evaluation comparison completed successfully!
